# 03 — Metric Construction

## Overview

This notebook implements the feature engineering stage of the higher education outcomes pipeline. It builds on the cleaned and integrated analytical base produced by `02_data_cleaning.ipynb`, without revisiting the audit, reconciliation, or integration decisions established there.

Feature engineering converts the institutional outcome counts into interpretable aggregate and ratio-based variables required for consistent comparisons across course sections with different enrollment levels.

The resulting dataset retains the cleaned institutional and operational descriptors while adding completion, attrition, adverse outcome, and excellence metrics. These variables constitute the analytical input for the statistical evaluation performed in `04_analysis.ipynb`.

## Load Cleaned Analytical Dataset

The consolidated dataset exported by `02_data_cleaning.ipynb` is loaded as the analytical base. It has already passed the preceding audit, cleaning, reconciliation, and integration stages; this notebook therefore assumes structurally consistent source variables and focuses exclusively on metric construction.

In [1]:
from notebook_utils import ensure_repo_root
ensure_repo_root()

WindowsPath('C:/Github/higher-education-outcomes-analysis')

In [2]:
import pandas as pd
import numpy as np

from src.utils.data_utils import load_data, set_pandas_display_options

data = load_data("demo base")

# Set pandas display options for better readability
set_pandas_display_options()

## Construct Analytical Metrics

The cleaned outcome components are transformed into aggregate counts and normalized rates. Together, these variables provide complementary representations of outcome volume and proportional performance for downstream analysis.

In [3]:
from src.metric_construction import (
    add_component_rates,
    add_columns_sum,
)

from src.config.columns import (
    SUM_COLS,
    METRIC_COMPONENT_COLS,
    NEW_RATE_METRIC_COLS,
)

### Aggregate Academic Outcome Counts

Related outcome components are combined into broader analytical categories: total completions, attrition, and adverse outcomes. These aggregates preserve the underlying institutional counts while providing concise measures for later outcome comparisons.

In [4]:
data_sum_metrics = data.copy()

for new_col, component_cols in SUM_COLS.items():
    data_sum_metrics = add_columns_sum(
        df=data_sum_metrics,
        columns_to_sum=component_cols,
        new_column_name=new_col,
        verbose=True
    )

Column `completion_count` created from: ['promoted_completion_count', 'regular_completion_count']
Column `attrition_count` created from: ['dropout_count', 'free_status_count']
Column `adverse_outcomes_count` created from: ['dropout_count', 'insufficient_count', 'free_status_count']


### Enrollment-Normalized Outcome Rates

Each component and aggregate outcome count is normalized by total enrollment. The resulting rates support comparisons across course sections of different sizes and form the principal outcome variables used in the statistical analysis stage.

In [5]:
data_rate_metrics = add_component_rates(
    df=data_sum_metrics,
    component_features=METRIC_COMPONENT_COLS,
    total_feature="total_enrollment",
    new_column_names=NEW_RATE_METRIC_COLS,
    verbose=True
)

Rate columns created against `total_enrollment`: ['dropout_rate', 'insufficient_rate', 'free_status_rate', 'promoted_completion_rate', 'regular_completion_rate', 'completion_rate', 'attrition_rate', 'adverse_outcomes_rate']


### Completion Composition

The `excellence_ratio` expresses promoted completions as a share of all completions. Unlike enrollment-normalized rates, this variable describes the internal composition of successful outcomes and distinguishes promoted completion from regular completion.

In [6]:
data_rate_metrics = add_component_rates(
    df=data_rate_metrics,
    component_features=["promoted_completion_count"],
    total_feature="completion_count",
    new_column_names=["excellence_ratio"],
    verbose=True
)

Rate columns created against `completion_count`: ['excellence_ratio']


Sections with no completions produce an undefined `excellence_ratio` because its denominator is zero. These observations are inspected before applying the established representation of zero for sections in which no promoted completions occurred.

In [7]:
display(data_rate_metrics[data_rate_metrics["excellence_ratio"].isna()])
display(data_rate_metrics[data_rate_metrics["completion_count"]==0])

,course_name,section,total_enrollment,dropout_count,insufficient_count,free_status_count,promoted_completion_count,regular_completion_count,course_code,program_code,workload,shift,weekday,schedule_time,delivery_mode,campus,program_name,completion_count,attrition_count,adverse_outcomes_count,dropout_rate,insufficient_rate,free_status_rate,promoted_completion_rate,regular_completion_rate,completion_rate,attrition_rate,adverse_outcomes_rate,excellence_ratio
288,COURSE 076,1,2,0,1,1,0,0,CRS_076,PRG_006,6,night,thursday,19 a 23,hybrid,CAMPUS_003,Program F,0,1,2,0.00,0.50,0.50,0.00,0.00,0.00,0.50,1.00,NaN


,course_name,section,total_enrollment,dropout_count,insufficient_count,free_status_count,promoted_completion_count,regular_completion_count,course_code,program_code,workload,shift,weekday,schedule_time,delivery_mode,campus,program_name,completion_count,attrition_count,adverse_outcomes_count,dropout_rate,insufficient_rate,free_status_rate,promoted_completion_rate,regular_completion_rate,completion_rate,attrition_rate,adverse_outcomes_rate,excellence_ratio
288,COURSE 076,1,2,0,1,1,0,0,CRS_076,PRG_006,6,night,thursday,19 a 23,hybrid,CAMPUS_003,Program F,0,1,2,0.00,0.50,0.50,0.00,0.00,0.00,0.50,1.00,NaN


In [8]:
data_rate_metrics["excellence_ratio"] = data_rate_metrics["excellence_ratio"].fillna(0)

## Validate Constructed Metrics

Lightweight assertions verify that the derived variables remain internally consistent with their source counts and expected analytical domains.

In [9]:
outcome_count_columns = [
    "dropout_count",
    "insufficient_count",
    "free_status_count",
    "promoted_completion_count",
    "regular_completion_count",
]
constructed_count_columns = list(SUM_COLS)
constructed_rate_columns = [*NEW_RATE_METRIC_COLS, "excellence_ratio"]
constructed_metric_columns = [
    *constructed_count_columns,
    *constructed_rate_columns,
]

assert not data_rate_metrics[constructed_metric_columns].isna().any().any(), (
    "Constructed metrics contain unexpected missing values."
)

assert (
    data_rate_metrics[["total_enrollment", *outcome_count_columns, *constructed_count_columns]] >= 0
).all().all(), "Outcome counts contain impossible negative values."

assert data_rate_metrics[constructed_rate_columns].apply(
    lambda column: column.between(0, 1).all()
).all(), "Constructed rates must remain within [0, 1]." # type: ignore

np.testing.assert_allclose(
    data_rate_metrics["completion_rate"],
    data_rate_metrics["promoted_completion_rate"]
    + data_rate_metrics["regular_completion_rate"],
    rtol=1e-10,
    atol=1e-12,
    err_msg=(
        "completion_rate must equal promoted_completion_rate plus "
        "regular_completion_rate."
    ),
)
np.testing.assert_array_equal(
    data_rate_metrics[outcome_count_columns].sum(axis=1),
    data_rate_metrics["total_enrollment"],
    err_msg="Outcome component counts must equal total_enrollment.",
)

pd.Series(
    {
        "no_missing_constructed_metrics": True,
        "non_negative_outcome_counts": True,
        "rates_within_unit_interval": True,
        "completion_rate_decomposition": True,
        "outcome_counts_match_enrollment": True,
    },
    name="validation_passed",
)

no_missing_constructed_metrics     True
non_negative_outcome_counts        True
rates_within_unit_interval         True
completion_rate_decomposition      True
outcome_counts_match_enrollment    True
Name: validation_passed, dtype: bool

All constructed metrics satisfy their expected bounds, additive identities, and completeness requirements. These checks provide confidence that the exported analytical dataset is internally consistent before inferential analysis.

## Final Analytical Dataset

The final schema retains the cleaned course, program, scheduling, and enrollment variables and appends the constructed outcome counts and rates. This unified section-level dataset provides the variables required for the next analytical stage without altering the source institutional structure.

A concise preview confirms the resulting column composition and representative values.

In [10]:
data_rate_metrics.head()

,course_name,section,total_enrollment,dropout_count,insufficient_count,free_status_count,promoted_completion_count,regular_completion_count,course_code,program_code,workload,shift,weekday,schedule_time,delivery_mode,campus,program_name,completion_count,attrition_count,adverse_outcomes_count,dropout_rate,insufficient_rate,free_status_rate,promoted_completion_rate,regular_completion_rate,completion_rate,attrition_rate,adverse_outcomes_rate,excellence_ratio
0,COURSE 090,1,110,5,4,4,43,54,CRS_090,PRG_013,6,night,thursday,19 a 23,online,CAMPUS_006,Program M,97,9,13,0.05,0.04,0.04,0.39,0.49,0.88,0.08,0.12,0.44
1,COURSE 046,1,98,13,8,7,34,36,CRS_046,PRG_002,6,night,tuesday,18 a 22,online,CAMPUS_006,Program B,70,20,28,0.13,0.08,0.07,0.35,0.37,0.71,0.20,0.29,0.49
2,COURSE 046,2,96,21,24,8,19,24,CRS_046,PRG_013,6,night,tuesday,18 a 22,online,CAMPUS_006,Program M,43,29,53,0.22,0.25,0.08,0.20,0.25,0.45,0.30,0.55,0.44
3,COURSE 046,3,112,10,14,10,25,53,CRS_046,PRG_002,6,night,wednesday,18 a 22,online,CAMPUS_006,Program B,78,20,34,0.09,0.12,0.09,0.22,0.47,0.70,0.18,0.30,0.32
4,COURSE 046,4,106,25,29,7,20,25,CRS_046,PRG_013,6,afternoon,thursday,14 a 18,online,CAMPUS_006,Program M,45,32,61,0.24,0.27,0.07,0.19,0.24,0.42,0.30,0.58,0.44


## Export Analytical Dataset

The validated dataset is persisted using the existing export path and schema. This artifact constitutes the analytical input consumed by `04_analysis.ipynb`.

In [11]:
from src.utils.io import export_parquet
from src.config.constants import DEMO_DATA_DIR
from pathlib import Path

path = Path(DEMO_DATA_DIR) / "demo_processed_data.parquet"

export_parquet(
    df=data_rate_metrics,
    path=path,
)

## Final Remarks

This feature engineering stage transformed the cleaned institutional outcome components into validated aggregate counts, enrollment-normalized rates, and a completion composition indicator. The resulting analytical dataset preserves the established source structure while providing internally consistent variables suitable for statistical inference.

The following notebook shifts from feature engineering to statistical inference, using the analytical dataset generated here to investigate the research questions introduced in the project and evaluate whether meaningful differences in academic outcomes emerge across key institutional dimensions.